In [1]:
import os
from pathlib import Path
import google.generativeai as genai
from dotenv import load_dotenv
from pprint import pprint


os.environ["TOKENIZERS_PARALLELISM"]="true"
load_dotenv()
genai.configure(api_key=os.environ.get("GOOGLE_API_KEY"))

In [2]:
VISION_MODEL = "gemini-pro-vision"
FILE_PATH = "./../test/image_001.jpg"
GENERATION_CONFIG = {
  "temperature": 0,
  "top_p": 1,
  "top_k": 32,
  "max_output_tokens": 4096,
}
SAFETY_SETTINGS = [
    {
        "category": "HARM_CATEGORY_HARASSMENT",
        "threshold": "BLOCK_NONE",
    },
    {
        "category": "HARM_CATEGORY_DANGEROUS_CONTENT",
        "threshold": "BLOCK_NONE",
    },
    {
        "category": "HARM_CATEGORY_HATE_SPEECH",
        "threshold": "BLOCK_NONE",
    },
    {
        "category": "HARM_CATEGORY_SEXUALLY_EXPLICIT",
        "threshold": "BLOCK_NONE",
    },
]


vision_model = genai.GenerativeModel(
    model_name=VISION_MODEL,
    generation_config=GENERATION_CONFIG,
    safety_settings=SAFETY_SETTINGS,
)


if not (img := Path(FILE_PATH)).exists():
    raise FileNotFoundError(f"Could not find image: {img}")


image_parts = [
    {
        "mime_type": "image/jpeg",
        "data": img.read_bytes()
    },
]

prompt_parts = [
    "Extract complete text in the image(s) below:\n",
    image_parts[0],
    "\nResponse: ",
]


response = vision_model.generate_content(prompt_parts)
pprint(response.text)

(" Người đầu tiên 'phá đảo' game xếp hình\n"
 'Blue Scuti, streamer 13 tuổi, trở thành người đầu tiên vượt qua toàn bộ màn '
 'chơi trong game Tetris trên hệ máy NES, còn được gọi là "Game xếp hình".\n'
 'Thành tích của Blue Scuti, tên thật là Willis Gibson, người Mỹ, được thực '
 'hiện ở trận bán kết Giải vô địch thế giới Tetris cổ điển (CTWC), diễn ra '
 'ngày 13/10/2023. Còn')


In [3]:
TEXT_MODEL = "gemini-pro"

text_model = genai.GenerativeModel(model_name = TEXT_MODEL)

In [4]:
ques = """
Ai là người lập kỷ lục, người đó làm nghề gì, quốc tịch gì, và bao nhiêu tuổi? Tìm kiếm thông tin trong Context sau.\n\n
Context:\n{}?n\
The output should be a json formatted in the following schema:\n
\"name\": string // Tên \n
\"occupation\": string // Nghề nghiệp \n
\"nationality\": string // Quốc tịch \n
\"age\": string // Tuổi \n
""".format(response.text)


response1 = text_model.generate_content(
    ques,
    generation_config=GENERATION_CONFIG,
    safety_settings=SAFETY_SETTINGS,
)


response1.text

'{\n "name": "Willis Gibson",\n "occupation": "Streamer",\n "nationality": "Mỹ",\n "age": "13"\n}'